In [28]:
import json
import math
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from scipy.stats import skew
from sklearn.metrics import roc_auc_score
from sentence_transformers import SentenceTransformer

BACKEND_DIR = Path("..").resolve()
KB_DIR = BACKEND_DIR / "knowledge_base"

KB_JSON = KB_DIR / "knowledge_base.json"

VAL_UPITI_JSON = KB_DIR / "tekst_upiti.json"
VAN_TEME_UPITI_JSON = KB_DIR / "tekst_upiti_van_teme.json"
AI_UPITI_JSON = KB_DIR / "tekst_upiti_ai.json"

EVAL_CSV = BACKEND_DIR / "rezultati/bge_large_rezultati.csv"
REZULTAT_CSV = BACKEND_DIR / "eval_text_deljenje_teksta.csv"

MODEL_NAME = "BAAI/bge-small-en-v1.5"
PREFIKS_UPITA = "Represent this sentence for searching relevant passages: "
BATCH_SIZE = 64
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

pd.set_option("display.width", 200)

In [29]:
with open(KB_JSON, encoding="utf-8") as f:
    knowledge_base = json.load(f)

with open(VAL_UPITI_JSON, encoding="utf-8") as f:
    val_upiti = json.load(f)

with open(VAN_TEME_UPITI_JSON, encoding="utf-8") as f:
    van_teme_upiti = json.load(f)["upiti"]

with open(AI_UPITI_JSON, encoding="utf-8") as f:
    ai_upiti = json.load(f)["upiti"]

specie_ids = sorted(knowledge_base)
BROJ_VRSTA = len(specie_ids)

INDEKS = {}
for specie_id, i in enumerate(specie_ids):
    INDEKS[i] = specie_id 

redovi = []
for sid in val_upiti:
    if sid == "_meta":
        continue
    for stavka in val_upiti[sid]["val"]:
        redovi.append({"specie_id": sid, "stil": stavka["stil"], "upit": stavka["upit"]})
val_df = pd.DataFrame(redovi)

tacne_vrste_val_i = np.array([INDEKS[sid] for sid in val_df["specie_id"]])  
tacne_vrste_ai_i = np.array([INDEKS[sid] for sid in ai_upiti])

novi_nacin_rezultati = pd.read_csv(EVAL_CSV)
assert list(novi_nacin_rezultati["upit"]) == list(val_df["upit"])
val_df["visak"] = novi_nacin_rezultati["visak"]


print("Vrsta:", BROJ_VRSTA, "| val upita:", len(val_df), "| upita van teme:", len(van_teme_upiti), "| ai upita:", len(ai_upiti))

Vrsta: 324 | val upita: 648 | upita van teme: 40 | ai upita: 18


In [30]:
model = SentenceTransformer(MODEL_NAME, device=DEVICE)
tokenizer = model.tokenizer

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1402.66it/s]


In [31]:
def ocisti_naslove(tekst):
    tekst = re.sub(r"\s*^=+[^\n]*=+\s*\Z", "", tekst, flags=re.MULTILINE)
    tekst = re.sub(r"^=+\s*(.+?)\s*=+[ \t]*\n\s*", r"\1: ", tekst, flags=re.MULTILINE)
    return tekst.strip()

ociscen_tekst = {}
for sid in specie_ids:
    ociscen_tekst[sid] = ocisti_naslove(knowledge_base[sid]["tekst"])

In [32]:
def broj_tokena(tekst):
    return len(tokenizer.encode(tekst, add_special_tokens=False))


KRAJ_RECENICE = r"(?<=[.!?])\s+(?=[A-Z])"
SEPARATORI = ["\n\n\n", "\n\n", "\n", KRAJ_RECENICE]

def podeli(tekst, separator):
    if separator == KRAJ_RECENICE:
        delovi = re.split(KRAJ_RECENICE, tekst)
    else:
        delovi = tekst.split(separator)
    rezultat = []
    for deo in delovi:
        if deo.strip() != "":
            rezultat.append(deo.strip())
    return rezultat

In [33]:

def podeli_na_chunkove(tekst, max_tokena=500, nivo=0):
    if broj_tokena(tekst) <= max_tokena or nivo == len(SEPARATORI):
        return [tekst]

    separator = SEPARATORI[nivo]
    celine = podeli(tekst, separator)
    if len(celine) == 1:
        return podeli_na_chunkove(tekst, max_tokena, nivo + 1)

    delovi = []
    for celina in celine:
        if broj_tokena(celina) <= max_tokena:
            delovi.append(celina)
        else:
            delovi += podeli_na_chunkove(celina, max_tokena, nivo + 1)

    if separator == KRAJ_RECENICE:
        spojnica = " "
    else:
        spojnica = separator
    return spoji_u_chunkove(delovi, spojnica, max_tokena)


def spoji_u_chunkove(celine, spojnica, max_tokena):
    ukupno = 0
    for celina in celine:
        ukupno += broj_tokena(celina)
    broj_delova = math.ceil(ukupno / max_tokena)
    cilj = ukupno / broj_delova

    chunkovi = []
    tekuci = []
    tekuci_tokeni = 0
    for celina in celine:
        n = broj_tokena(celina)
        if len(tekuci) > 0:
            novo = tekuci_tokeni + n
            if novo > max_tokena or abs(novo - cilj) > abs(tekuci_tokeni - cilj):
                chunkovi.append(spojnica.join(tekuci))
                tekuci = []
                tekuci_tokeni = 0
        tekuci.append(celina)
        tekuci_tokeni += n
    chunkovi.append(spojnica.join(tekuci))

    if len(chunkovi) > 1 and broj_tokena(chunkovi[-2]) + broj_tokena(chunkovi[-1]) <= max_tokena:
        poslednji = chunkovi.pop()
        chunkovi[-1] = chunkovi[-1] + spojnica + poslednji
    return chunkovi


def recenice(tekst):
    lista = []
    for red in podeli(tekst, "\n"):
        lista += podeli(red, KRAJ_RECENICE)
    return lista


def prozori(tekst, sirina=2, korak=1):
    r = recenice(tekst)
    if len(r) <= sirina:
        return [" ".join(r)]
    poceci = list(range(0, len(r) - sirina + 1, korak))
    if poceci[-1] != len(r) - sirina:   
        poceci.append(len(r) - sirina)
    lista = []
    for i in poceci:
        lista.append(" ".join(r[i:i + sirina]))
    return lista

In [34]:
def centriraj(v, mu):
    v = v - mu
    return v / np.linalg.norm(v, axis=-1, keepdims=True)

In [35]:
def embedovanje_teksta(chunkovi_ready):
    return model.encode(chunkovi_ready, batch_size=BATCH_SIZE, normalize_embeddings=True)

In [36]:
def podeli_tekst(tekst, nacin):
    if nacin == "do 500 tokena":
        return podeli_na_chunkove(tekst, 500)
    if nacin == "do 256 tokena":
        return podeli_na_chunkove(tekst, 256)
    if nacin == "do 128 tokena":
        return podeli_na_chunkove(tekst, 128)
    if nacin == "do 64 tokena":
        return podeli_na_chunkove(tekst, 64)
    if nacin == "jedna rečenica":
        return recenice(tekst)
    if nacin == "prozor 2 rečenice, bez preklapanja":
            return prozori(tekst, sirina=2, korak=2)
    if nacin == "prozor 2 rečenice":
            return prozori(tekst, sirina=2, korak=1)
    if nacin == "prozor 3 rečenice":
        return prozori(tekst, sirina=3, korak=1)
    raise ValueError(f"Nepoznat način: {nacin}")


def napravi_embeddings(nacin):
    delovi_svih_vrsta = []
    vrste = []
    for j, sid in enumerate(specie_ids):
        delovi = podeli_tekst(ociscen_tekst[sid], nacin)
        delovi_svih_vrsta += delovi
        vrste += [j] * len(delovi) 
    return embedovanje_teksta(delovi_svih_vrsta), np.array(vrste), delovi_svih_vrsta

In [37]:
def skor_vrste(sim, chunks_species):
    kolone = []
    for j in range(BROJ_VRSTA):
        kolone.append(sim[:, chunks_species == j].max(axis=1))
    return np.column_stack(kolone)


def rang_tacne(upiti_chunkovi, tacna_vrsta):
    rangovi = []
    for i in range(len(upiti_chunkovi)):
        poredak = list(np.argsort(-upiti_chunkovi[i]))
        rangovi.append(poredak.index(tacna_vrsta[i]) + 1)
    return np.array(rangovi)

def mere(S, val_vrste_i):
    rang = rang_tacne(S, val_vrste_i)

    rezultat = {}
    rezultat["Hit@1"] = np.mean(rang <= 1) * 100
    rezultat["Hit@3"] = np.mean(rang <= 3) * 100
    rezultat["Hit@10"] = np.mean(rang <= 10) * 100
    rezultat["MRR"] = np.mean(1 / rang)

    sortirano = np.sort(S, axis=1)
    rezultat["gap_srednja_vrednost"] = np.median(sortirano[:, -1] - sortirano[:, -2])

    prva = np.bincount(S.argmax(axis=1), minlength=BROJ_VRSTA)

    deset_posto = round(BROJ_VRSTA * 0.10)
    rezultat["10%_vrsta_uzima_%"] = np.sort(prva)[::-1][:deset_posto].sum() / len(S) * 100
    
    rezultat["nikad_prve"] = int((prva == 0).sum())

    for k in [1, 5, 10]:
        prvih_k = np.argsort(-S, axis=1)[:, :k]
        N_k = np.bincount(prvih_k.ravel(), minlength=BROJ_VRSTA)
        rezultat[f"skew_N{k}"] = skew(N_k)
        rezultat[f"antihub_N{k}"] = int((N_k == 0).sum())
    return rezultat

In [38]:
sirovi_vektori_val_upita = embedovanje_teksta([PREFIKS_UPITA + u for u in val_df["upit"]])
srednji_val_vektor = sirovi_vektori_val_upita.mean(axis=0)
centrirani_vektori_val_upita = centriraj(sirovi_vektori_val_upita, srednji_val_vektor)

In [39]:
duzina_teksta_reci = pd.Series({sid: len(ociscen_tekst[sid].split()) for sid in specie_ids})
duzina_teksta_tokeni = pd.Series({sid: broj_tokena(ociscen_tekst[sid]) for sid in specie_ids})

print(duzina_teksta_tokeni.describe().round(0))
print(f"\nOpisa do 500 tokena (ostaju jedan chunk): {(duzina_teksta_tokeni <= 500).sum()} od {BROJ_VRSTA}")

chunks, chunks_species, _ = napravi_embeddings("do 500 tokena")
S_ranije = skor_vrste(sirovi_vektori_val_upita @ chunks.T, chunks_species)

print("\nChunk-ova:", len(chunks))
prvo_testiranje = pd.Series(mere(S_ranije, tacne_vrste_val_i)).round(3)
prvo_testiranje.to_frame("vrednost").to_csv(BACKEND_DIR / "rezultati" / "prvo_testiranje_mere.csv",
                                            index_label="mera", encoding="utf-8-sig")
print(prvo_testiranje)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1108 > 512). Running this sequence through the model will result in indexing errors


count     324.0
mean      220.0
std       173.0
min        16.0
25%       110.0
50%       169.0
75%       282.0
max      1108.0
dtype: float64

Opisa do 500 tokena (ostaju jedan chunk): 304 od 324

Chunk-ova: 349
Hit@1                    20.988
Hit@3                    30.401
Hit@10                   45.062
MRR                       0.288
gap_srednja_vrednost      0.012
10%_vrsta_uzima_%        71.759
nikad_prve              204.000
skew_N1                   4.974
antihub_N1              204.000
skew_N5                   4.482
antihub_N5              101.000
skew_N10                  3.797
antihub_N10              51.000
dtype: float64


In [40]:
rang_ranije = rang_tacne(S_ranije, tacne_vrste_val_i)
duzina_upita = val_df["specie_id"].map(duzina_teksta_reci)
kvartil = pd.qcut(duzina_upita, 4, duplicates="drop")
hit1 = pd.Series(rang_ranije <= 1, index=val_df.index)
hit_po_duzini = (
    hit1
    .groupby(kvartil, observed=True)
    .mean()
    .mul(100)
    .round(1)
)

print("Hit@1 po dužini opisa tačne vrste:")
for interval, procenat in hit_po_duzini.items():
    print(f"{interval.left:.0f}–{interval.right:.0f} reči: {procenat:.1f}%")

razvodnjavanje = pd.DataFrame({
    "reci_od": [interval.left for interval in hit_po_duzini.index],
    "reci_do": [interval.right for interval in hit_po_duzini.index],
    "Hit@1_%": hit_po_duzini.values,
})
razvodnjavanje.to_csv(BACKEND_DIR / "rezultati" / "prvo_testiranje_razvodnjavanje.csv", index=False, encoding="utf-8-sig")

prva = pd.Series(np.bincount(S_ranije.argmax(axis=1), minlength=BROJ_VRSTA), index=specie_ids)
najcesce = prva.nlargest(8).index
hubness = pd.DataFrame({
    "scientific_name": [knowledge_base[s]["scientific_name"] for s in najcesce],
    "puta_prva": prva[najcesce],
    "reci_u_opisu": duzina_teksta_reci[najcesce],
})
hubness.to_csv(BACKEND_DIR / "rezultati" / "prvo_testiranje_hubness.csv", index_label="specie_id", encoding="utf-8-sig")
print("\nNajčešće prve vrste:")
print(hubness)

Hit@1 po dužini opisa tačne vrste:
12–74 reči: 46.3%
74–113 reči: 14.2%
113–185 reči: 9.9%
185–806 reči: 13.1%

Najčešće prve vrste:
             scientific_name  puta_prva  reci_u_opisu
0240     Guizotia abyssinica         45            68
0222        Trifolium dubium         39            39
0554       Hypericum patulum         39           142
0032     Gypsophila vaccaria         34            38
0497  Petrosedum ochroleucum         33            37
0728   Maianthemum canadense         29           216
0010    Phyllanthus tenellus         26            49
0932   Crassothonna capensis         24            32


In [27]:
VARIJANTE = ["do 500 tokena", "do 256 tokena", "do 128 tokena", "do 64 tokena",
             "jedna rečenica", "prozor 3 rečenice", "prozor 2 rečenice"]

CHUNKOVI = {}
redovi = []
for nacin in VARIJANTE:
    sirovi_vektori, species_i, species_opisi = napravi_embeddings(nacin)
    CHUNKOVI[nacin] = (sirovi_vektori, species_i)
    centrirani_vektori = centriraj(sirovi_vektori, sirovi_vektori.mean(axis=0))

    bez = mere(skor_vrste(sirovi_vektori_val_upita @ sirovi_vektori.T, species_i), tacne_vrste_val_i)
    sa = mere(skor_vrste(centrirani_vektori_val_upita @ centrirani_vektori.T, species_i), tacne_vrste_val_i)
   
    tokena = int(np.median([broj_tokena(t) for t in species_opisi]))

    redovi.append({"chunk": nacin, "centrirano": "ne", "chunk-ova": len(sirovi_vektori), "tokena_medijana": tokena, **bez})
    redovi.append({"chunk": nacin, "centrirano": "da", "chunk-ova": len(sirovi_vektori), "tokena_medijana": tokena, **sa})

drugo_testiranje = pd.DataFrame(redovi).set_index(["chunk", "centrirano"]).round(3)
drugo_testiranje.to_csv(BACKEND_DIR / "rezultati" / "drugo_testiranje_chunkovi.csv", encoding="utf-8-sig")
drugo_testiranje

chunk-ova  tokena_medijana   Hit@1   Hit@3  Hit@10    MRR  gap_srednja_vrednost  10%_vrsta_uzima_%  nikad_prve  skew_N1  antihub_N1  skew_N5  antihub_N5  skew_N10  \
chunk             centrirano                                                                                                                                                                       
do 500 tokena     ne                349              175  20.988  30.401  45.062  0.288                 0.012             71.759         204    4.974         204    4.482         101     3.797   
                  da                349              175  27.623  41.667  59.877  0.380                 0.035             45.062         128    2.567         128    1.940          11     1.577   
do 256 tokena     ne                470              153  21.759  31.636  46.914  0.301                 0.012             68.210         187    4.853         187    4.310          92     3.407   
                  da                470              153  28.549  43.210  60.802  0.391                 0.035             43.827         121    2.411         121    2.044          19     1.480   
do 128 tokena     ne                823               88  26.389  38.889  53.395  0.355                 0.012             56.944         155    3.707         155    2.762          68     2.326   
                  da                823               88  33.333  48.611  63.580  0.439                 0.034             36.420          94    1.709          94    1.893          22     1.887   
do 64 tokena      ne               1508               48  27.623  41.049  58.488  0.380                 0.013             54.475         133    3.923         133    2.925          36     2.388   
                  da               1508               48  36.728  52.006  70.679  0.476                 0.034             32.253          76    1.460          76    1.447          11     1.398   
jedna rečenica    ne               2821               22  25.154  37.809  54.630  0.354                 0.014             55.401         136    8.760         136    3.877          28     2.810   
                  da               2821               22  39.660  56.019  72.222  0.508                 0.035             30.247          66    1.842          66    1.293           8     1.280   
prozor 3 rečenice ne               2216               69  33.179  47.531  63.272  0.433                 0.012             45.370         114    2.833         114    2.308          38     2.206   
                  da               2216               69  40.278  56.327  74.074  0.514                 0.041             32.407          71    1.871          71    1.541          14     1.553   
prozor 2 rečenice ne               2509               46  31.790  49.383  65.741  0.436                 0.013             46.759         118    2.921         118    2.460          26     2.013   
                  da               2509               46  39.815  59.105  73.611  0.519                 0.039             30.864          72    1.492          72    1.078          10     1.194   

                              antihub_N10  
chunk             centrirano               
do 500 tokena     ne                   51  
                  da                    0  
do 256 tokena     ne                   61  
                  da                    1  
do 128 tokena     ne                   42  
                  da                    7  
do 64 tokena      ne                   22  
                  da                    3  
jedna rečenica    ne                   11  
                  da                    1  
prozor 3 rečenice ne                   29  
                  da                    6  
prozor 2 rečenice ne                   18  
                  da                    2

In [ ]:
sirovi_vektori, prozori_i = CHUNKOVI["prozor 2 rečenice"]
centrirani_vektori = centriraj(sirovi_vektori, sirovi_vektori.mean(axis=0))
skalarni_proizvodi = centrirani_vektori_val_upita @ centrirani_vektori.T

T = 0.05


def agregiraj(sim, prozori_i, nacin):
    kolone = []
    for j in range(BROJ_VRSTA):
        s = sim[:, prozori_i == j]
        if nacin == "prosek top2":
            kolone.append(np.sort(s, axis=1)[:, -2:].mean(axis=1))
        elif nacin == "logmeanexp":
            kolone.append(T * np.log(np.exp(s / T).mean(axis=1)))
    return np.column_stack(kolone)


agregacija = pd.DataFrame({
    "max": mere(skor_vrste(skalarni_proizvodi, prozori_i), tacne_vrste_val_i),
    "prosek top2": mere(agregiraj(skalarni_proizvodi, prozori_i, "prosek top2"), tacne_vrste_val_i),
    "logmeanexp": mere(agregiraj(skalarni_proizvodi, prozori_i, "logmeanexp"), tacne_vrste_val_i),
}).T.round(3)
agregacija.to_csv(BACKEND_DIR / "rezultati" / "agregacija.csv", index_label="agregacija", encoding="utf-8-sig")
agregacija

,Hit@1,Hit@3,Hit@10,MRR,gap_srednja_vrednost,10%_vrsta_uzima_%,nikad_prve,skew_N1,antihub_N1,skew_N5,antihub_N5,skew_N10,antihub_N10
max,39.815,59.105,73.611,0.519,0.039,30.864,72.0,1.492,72.0,1.078,10.0,1.194,2.0
prosek top2,41.512,59.414,74.846,0.531,0.033,34.414,71.0,2.295,71.0,1.844,16.0,1.813,5.0
logmeanexp,43.056,60.648,76.698,0.545,0.044,33.333,81.0,2.292,81.0,1.303,4.0,0.827,0.0


In [42]:
centrirani_dokumenti = centriraj(chunks, chunks.mean(axis=0))

skorovi_dokumenata = skor_vrste(centrirani_vektori_val_upita @ centrirani_dokumenti.T, chunks_species)


def konacni_skor(sirovi_prozori, prozori_i, tezina_dokumenta=0.3):
    centrirani_prozori = centriraj(sirovi_prozori, sirovi_prozori.mean(axis=0))
    skorovi_prozora = skor_vrste(centrirani_vektori_val_upita @ centrirani_prozori.T, prozori_i)

    return (1 - tezina_dokumenta) * skorovi_prozora + tezina_dokumenta * skorovi_dokumenata


sa_preklapanjem, sa_preklapanjem_i = CHUNKOVI["prozor 2 rečenice"]

bez_preklapanja, bez_preklapanja_i, _ = napravi_embeddings("prozor 2 rečenice, bez preklapanja")

preklapanje = pd.DataFrame({
    "sa preklapanjem": {"chunk-ova": len(sa_preklapanjem),
                        **mere(konacni_skor(sa_preklapanjem, sa_preklapanjem_i), tacne_vrste_val_i)},
    "bez preklapanja": {"chunk-ova": len(bez_preklapanja),
                        **mere(konacni_skor(bez_preklapanja, bez_preklapanja_i), tacne_vrste_val_i)},
}).T.round(3)
preklapanje.to_csv(BACKEND_DIR / "rezultati" / "preklapanje_prozora.csv", index_label="prozori", encoding="utf-8-sig")
preklapanje

,chunk-ova,Hit@1,Hit@3,Hit@10,MRR,gap_srednja_vrednost,10%_vrsta_uzima_%,nikad_prve,skew_N1,antihub_N1,skew_N5,antihub_N5,skew_N10,antihub_N10
sa preklapanjem,2509.0,43.827,63.272,77.315,0.557,0.037,31.327,69.0,1.933,69.0,1.632,4.0,1.486,0.0
bez preklapanja,1494.0,38.735,56.327,72.222,0.505,0.036,35.648,92.0,2.000,92.0,1.661,6.0,1.654,0.0


In [43]:
tezine = {}
for w in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.7, 1.0]:
    tezine[w] = mere(konacni_skor(sa_preklapanjem, sa_preklapanjem_i, w), tacne_vrste_val_i)

tezina_dokumenta = pd.DataFrame(tezine).T.round(3)
tezina_dokumenta.index.name = "w"
tezina_dokumenta.to_csv(BACKEND_DIR / "rezultati" / "tezina_dokumenta.csv", encoding="utf-8-sig")
tezina_dokumenta

,Hit@1,Hit@3,Hit@10,MRR,gap_srednja_vrednost,10%_vrsta_uzima_%,nikad_prve,skew_N1,antihub_N1,skew_N5,antihub_N5,skew_N10,antihub_N10
w,,,,,,,,,,,,,
0.0,39.815,59.105,73.611,0.519,0.039,30.864,72.0,1.492,72.0,1.078,10.0,1.194,2.0
0.1,41.512,61.420,75.926,0.536,0.039,30.710,68.0,1.800,68.0,1.195,7.0,1.248,1.0
0.2,42.901,62.654,77.006,0.551,0.041,31.327,70.0,1.713,70.0,1.371,6.0,1.348,0.0
0.3,43.827,63.272,77.315,0.557,0.037,31.327,69.0,1.933,69.0,1.632,4.0,1.486,0.0
0.4,42.747,61.420,77.160,0.547,0.033,34.568,81.0,1.996,81.0,1.882,3.0,1.847,0.0
0.5,40.895,60.185,75.154,0.532,0.032,37.654,87.0,2.410,87.0,2.050,3.0,1.934,0.0
0.7,34.414,53.395,70.370,0.470,0.034,42.130,109.0,2.410,109.0,1.992,7.0,1.737,0.0
1.0,27.623,41.667,59.877,0.380,0.035,45.062,128.0,2.567,128.0,1.940,11.0,1.577,0.0


In [44]:
centrirani_prozori = centriraj(sa_preklapanjem, sa_preklapanjem.mean(axis=0))
skorovi_upit_prozor = centrirani_vektori_val_upita @ centrirani_prozori.T

skorovi_max = skor_vrste(skorovi_upit_prozor, sa_preklapanjem_i)
skorovi_lme = agregiraj(skorovi_upit_prozor, sa_preklapanjem_i, "logmeanexp")
pogodak_max = rang_tacne(0.7 * skorovi_max + 0.3 * skorovi_dokumenata, tacne_vrste_val_i) <= 1
pogodak_lme = rang_tacne(0.7 * skorovi_lme + 0.3 * skorovi_dokumenata, tacne_vrste_val_i) <= 1

rng = np.random.default_rng(42)
indeksi_po_vrsti = val_df.groupby("specie_id").indices
vrste = list(indeksi_po_vrsti)

razlike = []
for b in range(2000):
    uzorak_vrsta = rng.choice(vrste, len(vrste))
    indeksi = np.concatenate([indeksi_po_vrsti[v] for v in uzorak_vrsta])
    razlike.append((pogodak_lme[indeksi].mean() - pogodak_max[indeksi].mean()) * 100)

razlika = (pogodak_lme.mean() - pogodak_max.mean()) * 100
donja, gornja = np.percentile(razlike, 2.5), np.percentile(razlike, 97.5)

print(f"Hit@1: max {pogodak_max.mean() * 100:.1f}% | logmeanexp {pogodak_lme.mean() * 100:.1f}%")
print(f"razlika: {razlika:.1f} pp, 95% interval [{donja:.1f}, {gornja:.1f}]")

agregacija_uz_dokument = pd.DataFrame([{
    "Hit@1_max": pogodak_max.mean() * 100,
    "Hit@1_logmeanexp": pogodak_lme.mean() * 100,
    "razlika_pp": razlika,
    "interval_95_donja": donja,
    "interval_95_gornja": gornja,
    "bootstrap_uzoraka": len(razlike),
}]).round(2)
agregacija_uz_dokument.to_csv(BACKEND_DIR / "rezultati" / "agregacija_uz_dokument.csv", index=False, encoding="utf-8-sig")
agregacija_uz_dokument

Hit@1: max 43.8% | logmeanexp 43.4%
razlika: -0.5 pp, 95% interval [-2.9, 2.0]


,Hit@1_max,Hit@1_logmeanexp,razlika_pp,interval_95_donja,interval_95_gornja,bootstrap_uzoraka
0,43.83,43.36,-0.46,-2.93,2.01,2000
